<a href="https://colab.research.google.com/github/Hamerson-jhoel/Procesamiento-de-Lengaje-Natural/blob/main/CLASE9_LNP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# los RAG modelo de lengiaje

- R recuperacion en lenguja enatural se busca poara procesar y extraer datos de una base de conocimientos, norma interna de una empresa, contenidos de profesor
- A los fragmentos recupoerados en el paso anterior se inyecta directamente en el contexto , si le suministro base de informacion segun el contexto de la informacion dada el me dara lo que le pida  kernel en matematicas y en programacion ejemplo
- G disminuimos lo que se denomina alucinacion que ell modelo me entrege algo que no es real, la informacion puede ser coherente y creible epero el modelo me esta dando informacion falsa



In [24]:
!pip intall tranformers sentece-transformers langchain-text-splitters -q
!pip install google-genai -q
!pip install faiss-cpu -q

ERROR: unknown command "intall" - maybe you meant "install"


In [25]:
import numpy as np
import os

import torch

import faiss

from google import genai
from google.genai import types
from google.colab import userdata

from sentence_transformers import SentenceTransformer

Así es, estás en lo correcto. En la similitud coseno, los valores van desde \(-1\) hasta \(1\).
- Te explico exactamente qué significa cada valor:1 (Equivalentes / Idénticos): El ángulo entre los dos vectores es de \(0^{\circ }\). Esto significa que los textos o datos tienen exactamente la misma orientación y proporción, por lo que se consideran idénticos o equivalentes.
- 0 (Totalmente opuestos / Ortogonales): El ángulo entre los vectores es de \(90^{\circ }\). No comparten ninguna dirección en común; son independientes o totalmente diferentes.

In [26]:
device = "cuda" if torch.cuda.is_available() else "cpu"

embedding_model = SentenceTransformer("all-MiniLM-L6-v2").to(device=device)

frases=[
    "el sistema de frenos del coche fallo en la autopista.",
    "los componentes del motor muestran un desgaste prematuro",
    "ayer compre un almuerzo delicioso en la cafeteria",
    "el mecanismo de frenado no respondio correctamente"

]

with torch.no_grad():
  vectors = embedding_model.encode(frases, convert_to_tensor=True).to(device)

print(vectors.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([4, 384])


In [27]:
def calcular_similitud_coseno(tensor_a, tensor_b):


  producto_escalar = torch.dot(tensor_a, tensor_b)

  norma_a = torch.norm(tensor_a)
  norma_b = torch.norm(tensor_b)

  similitud_coseno = producto_escalar / (norma_a * norma_b)
  return similitud_coseno.item()

In [28]:
frase_objetivo = vectors[0]
for i, vector_frase in enumerate(vectors):
  similitud = calcular_similitud_coseno(frase_objetivo, vector_frase)
  print(f"frase {i+1}: '{frases[i]}'- {similitud}")

frase 1: 'el sistema de frenos del coche fallo en la autopista.'- 1.0000001192092896
frase 2: 'los componentes del motor muestran un desgaste prematuro'- 0.3673657476902008
frase 3: 'ayer compre un almuerzo delicioso en la cafeteria'- 0.5001679062843323
frase 4: 'el mecanismo de frenado no respondio correctamente'- 0.5472562313079834


#otro embedimiento, recibe texto , imagenes , audio es multimodal , de google

In [29]:
try:
  api_key = userdata.get('GEMINI_API_KEY')

  os.environ['GEMINI_API_KEY'] = api_key

  print("API key de google AI studio cargada.")


except Exception as e:

  print(f"Error al cargar la API key de Google AI Studio: {e}")

client = genai.Client()


API key de google AI studio cargada.


In [30]:
documento_tecnico = (
    "El sistema de control de estabilidad del vehículo monitorea constantemente la velocidad de las ruedas. "
    "Cuando la unidad de control electrónico (ECU) detecta una pérdida de tracción en el eje delantero, "
    "activa inmediatamente el módulo hidráulico del ABS para aplicar presión de frenado selectiva en la rueda trasera opuesta. "
    "Este ajuste físico estabiliza la guiñada y previene el subviraje crítico en curvas de alta velocidad. "
    "Adicionalmente, el sistema reduce el torque del motor de forma temporal interviniendo la mariposa de admisión. "
    "Para el mantenimiento de estos módulos, es mandatorio inspeccionar los sensores magnéticos de efecto Hall colocados en cada maza. "
    "Cualquier acumulación de residuos ferrosos en el anillo reluctor puede provocar lecturas erróneas de aceleración angular, "
    "lo que disparará una alerta en el panel de instrumentos desactivando las asistencias de seguridad de forma preventiva."
)


def segmentar_texto(texto,chunk_size=30, overlap=10):

  palabras = texto.split()
  chunks = []
  i = 0

  while i < len(palabras):
    ventana = palabras[i : i+chunk_size]

    fragmento = " ".join(ventana)
    chunks.append(fragmento)
    i += chunk_size - overlap

    if i + chunk_size > len(palabras) and i < len(palabras):
      restantes = palabras[i:]
      chunks.append(" ".join(restantes))
      break

  return chunks

In [31]:
fragmentos_procesados = segmentar_texto(documento_tecnico, chunk_size=25, overlap=8)

print(f"Número de fragmentos: {len(fragmentos_procesados)}")

for idx, chunk in enumerate(fragmentos_procesados):
  print(f"Fragmento {idx + 1}: (longitud: {len(chunk.split())} palabras:)\n{chunk}\n" )

Número de fragmentos: 8
Fragmento 1: (longitud: 25 palabras:)
El sistema de control de estabilidad del vehículo monitorea constantemente la velocidad de las ruedas. Cuando la unidad de control electrónico (ECU) detecta una pérdida

Fragmento 2: (longitud: 25 palabras:)
unidad de control electrónico (ECU) detecta una pérdida de tracción en el eje delantero, activa inmediatamente el módulo hidráulico del ABS para aplicar presión de

Fragmento 3: (longitud: 25 palabras:)
módulo hidráulico del ABS para aplicar presión de frenado selectiva en la rueda trasera opuesta. Este ajuste físico estabiliza la guiñada y previene el subviraje

Fragmento 4: (longitud: 25 palabras:)
físico estabiliza la guiñada y previene el subviraje crítico en curvas de alta velocidad. Adicionalmente, el sistema reduce el torque del motor de forma temporal

Fragmento 5: (longitud: 25 palabras:)
reduce el torque del motor de forma temporal interviniendo la mariposa de admisión. Para el mantenimiento de estos módulos, e

In [32]:
try:


  response = client.models.embed_content(
      model = 'gemini-embedding-2',
      contents = fragmentos_procesados
  )

  matriz_embeddings = [emb.values for emb in response.embeddings]

  print(len(matriz_embeddings))
  print(len(matriz_embeddings[0]))

  print(matriz_embeddings[0][:5])

except Exception as e:

  print(f"error al generar los embeddings: {e}")

8
3072
[0.0042387526, 0.001836521, -0.006040555, 0.004310009, 0.029615657]


# crear familias semnaticas para luego comparar con ellas, comparamos muestra contra familia

In [33]:
vectores_np = np.array(matriz_embeddings).astype('float32')

dimension_espacial = vectores_np.shape[1]

index = faiss.IndexFlatIP(dimension_espacial)

index.add(vectores_np)

In [34]:
pregunta_usuario = "¿Que componente fisico se debe revisar si el panel muestra un error magnetico?"

response_query = client.models.embed_content(
    model='gemini-embedding-2',
    contents = pregunta_usuario
)

vector_query = np.array([response_query.embeddings[0].values]).astype('float32')


In [35]:
K = 2
similitudes, indices_retornados= index.search(vector_query, K)

for i in range(K):
  idx_fragmento = indices_retornados[0][i]
  score_similitud = similitudes[0][i]
  texto_recuperado = fragmentos_procesados[idx_fragmento]

  print(idx_fragmento, score_similitud, texto_recuperado, sep='\n')

5
0.68799114
estos módulos, es mandatorio inspeccionar los sensores magnéticos de efecto Hall colocados en cada maza. Cualquier acumulación de residuos ferrosos en el anillo reluctor puede
4
0.6849713
reduce el torque del motor de forma temporal interviniendo la mariposa de admisión. Para el mantenimiento de estos módulos, es mandatorio inspeccionar los sensores magnéticos


In [36]:
contexto_consolidado = ""

for idx, i_index in enumerate(indices_retornados[0]):

  texto_fragmento = fragmentos_procesados[i_index]
  contexto_consolidado += f"--- Fragmento {idx+i} ---\n{texto_fragmento}\n"

print(contexto_consolidado)


--- Fragmento 1 ---
estos módulos, es mandatorio inspeccionar los sensores magnéticos de efecto Hall colocados en cada maza. Cualquier acumulación de residuos ferrosos en el anillo reluctor puede
--- Fragmento 2 ---
reduce el torque del motor de forma temporal interviniendo la mariposa de admisión. Para el mantenimiento de estos módulos, es mandatorio inspeccionar los sensores magnéticos



In [37]:
# 2. Diseñar el System Prompt restrictivo para mitigar alucinaciones
system_instruction = (
    "Eres un asistente de ingeniería experto y riguroso. Tu única tarea es responder "
    "preguntas técnicas basadas exclusivamente en el contexto provisto entre las etiquetas <context> y </context>.\n"
    "REGLAS OBLIGATORIAS:\n"
    "1. Responde de forma concisa, técnica y directa.\n"
    "2. Si la respuesta no se encuentra explícitamente en los fragmentos del contexto, debes contestar "
    "exactamente: 'Lo siento, la información solicitada no se encuentra en la documentación provista.'\n"
    "3. No inventes datos, no asumas conclusiones fuera del texto y bajo ninguna circunstancia uses conocimiento externo."
)

# 3. Estructurar el prompt de usuario inyectando el contexto y la pregunta original
user_prompt = f"""
Analiza los siguientes fragmentos técnicos e identifica la solución a la consulta.

<context>
{contexto_consolidado}
</context>

Consulta del usuario: {pregunta_usuario}
"""

In [38]:
user_prompt

'\nAnaliza los siguientes fragmentos técnicos e identifica la solución a la consulta.\n\n<context>\n--- Fragmento 1 ---\nestos módulos, es mandatorio inspeccionar los sensores magnéticos de efecto Hall colocados en cada maza. Cualquier acumulación de residuos ferrosos en el anillo reluctor puede\n--- Fragmento 2 ---\nreduce el torque del motor de forma temporal interviniendo la mariposa de admisión. Para el mantenimiento de estos módulos, es mandatorio inspeccionar los sensores magnéticos\n\n</context>\n\nConsulta del usuario: ¿Que componente fisico se debe revisar si el panel muestra un error magnetico?\n'

In [39]:
try:

  response_rag = client.models.generate_content(
      model = 'gemini-2.5-flash',
      contents = user_prompt,
      config=types.GenerateContentConfig(
          system_instruction = system_instruction,
          temperature = 0.0,
          max_output_tokens = 500
      )
  )
except Exception as e:
  print(f"error al generar la respuesta: {e}")

In [40]:
print(response_rag.text)

Se deben inspeccionar los sensores magnéticos de efecto Hall y el anillo reluctor.


In [41]:
pregunta_trampa = "¿Cuál es el torque de apriete recomendado para las tuercas de las llantas de aleación?"
print(f"Consulta trampa del usuario: '{pregunta_trampa}'")

response_trampa_emb = client.models.embed_content(
    model="gemini-embedding-2",
    contents=pregunta_trampa
)
vector_trampa = np.array([response_trampa_emb.embeddings[0].values]).astype('float32')

# Ejecutar la búsqueda en FAISS (traerá lo menos lejano numéricamente)
_, indices_trampa = index.search(vector_trampa, K)

contexto_trampa = ""
for idx, i_index in enumerate(indices_trampa[0]):
    contexto_trampa += f"--- Fragmento {idx+1} ---\n{fragmentos_procesados[i_index]}\n"

user_prompt_trampa = f"""
Analiza los siguientes fragmentos técnicos e identifica la solución a la consulta.

<context>
{contexto_trampa}
</context>

Consulta del usuario: {pregunta_trampa}
"""

# Ejecutar la inferencia con gemini-2.5-flash y la instrucción del sistema restrictiva
try:
    response_trampa = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=user_prompt_trampa,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            temperature=0.0
        )
    )
    print("\n--- RESPUESTA DEL SISTEMA ANTE PREGUNTA FUERA DE CONTEXTO ---")
    print(response_trampa.text)
    print("\nÉxito de ingeniería: El modelo bloqueó la alucinación respondiendo bajo las directrices del System Prompt.")

except Exception as e:
    print(f"Error en la prueba de mitigación: {e}")

Consulta trampa del usuario: '¿Cuál es el torque de apriete recomendado para las tuercas de las llantas de aleación?'

--- RESPUESTA DEL SISTEMA ANTE PREGUNTA FUERA DE CONTEXTO ---
Lo siento, la información solicitada no se encuentra en la documentación provista.

Éxito de ingeniería: El modelo bloqueó la alucinación respondiendo bajo las directrices del System Prompt.
